In [ ]:
## Load required libraries
!pip install -q transformers==4.39.3 datasets sentencepiece sacremoses evaluate rouge-score


In [ ]:
## Check GPU availability
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate


In [ ]:
## Set device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


In [ ]:
#login on hugging face
!pip install -q huggingface_hub


In [ ]:
from huggingface_hub import login

login()


In [ ]:
#login to base model
BASE_MODEL_ID = "ai4bharat/indictrans2-en-indic-dist-200M"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True
)


In [ ]:
## Load fine-tuned model and tokenizer
FINETUNED_MODEL_ID = "deepanshumiglani0408/indictrans2_finetune"

model = AutoModelForSeq2SeqLM.from_pretrained(
    FINETUNED_MODEL_ID,
    trust_remote_code=True,
    low_cpu_mem_usage=False   # 🔴 KEY LINE
)


In [ ]:
#removing cache
model = model.to(device)
model.config.use_cache = False
model.eval()


In [ ]:
## Define translation function
SRC_LANG = "eng_Latn"
TGT_LANG = "hin_Deva"

def translate(text):
    tagged = f"{SRC_LANG} {TGT_LANG} {text}"
    inputs = tokenizer(tagged, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=5,
            max_length=128,
            use_cache=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

  ## Test translation with a sample sentence


print(translate("A black box in your car?"))


In [ ]:
def translate_sampling(
    text,
    temperature=1.0,
    top_k=50,
    top_p=0.9
):
    tagged = f"{SRC_LANG} {TGT_LANG} {text}"
    inputs = tokenizer(tagged, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            num_beams=1,
            use_cache=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
test_sentence = "A black box in your car?, "


print("\nSampling (temperature=1.0):")
print(translate_sampling(test_sentence, temperature=1.0))


In [ ]:
##curating data of diffrent type of sentences
test_sentences = [
    # Short sentences (1–15)
    "Artificial Intelligence is transforming healthcare.",
    "India has a rich cultural heritage.",
    "Technology is evolving very rapidly.",
    "Education plays a vital role in development.",
    "Climate change is a global challenge.",
    "The government launched a new digital initiative.",
    "Renewable energy is essential for the future.",
    "Data quality impacts machine learning performance.",
    "Healthcare accessibility remains a major concern.",
    "Language translation enables global communication.",
    "Scientific research drives innovation.",
    "The economy depends on sustainable growth.",
    "Digital payments are widely used in India.",
    "Automation improves operational efficiency.",
    "Cybersecurity is critical in the digital age.",

    # Medium-length sentences (16–30)
    "Artificial Intelligence is helping doctors diagnose diseases more accurately and efficiently.",
    "India’s education system is gradually adopting digital tools to improve learning outcomes.",
    "Climate change affects agriculture, water resources, and public health worldwide.",
    "Machine learning models require large amounts of high-quality training data.",
    "The government introduced new policies to support startups and innovation.",
    "Renewable energy sources such as solar and wind are becoming more affordable.",
    "Digital platforms have transformed the way people communicate and work.",
    "Healthcare systems must balance affordability with quality medical services.",
    "Language translation tools help businesses expand into international markets.",
    "Technological advancements are reshaping traditional industries.",
    "Data privacy has become a major concern in online platforms.",
    "Education systems must adapt to meet the demands of the digital economy.",
    "Public infrastructure development plays a key role in economic growth.",
    "Artificial Intelligence applications are increasing across multiple sectors.",
    "Sustainable development requires cooperation between governments and industries.",

    # Long / complex sentences (31–45)
    "Artificial Intelligence is transforming the healthcare sector by enabling faster diagnosis, personalized treatment plans, and efficient management of medical resources across hospitals and rural clinics.",
    "India has a rich cultural heritage that spans thousands of years, reflecting a diverse mix of languages, traditions, festivals, and historical influences that continue to shape modern society.",
    "Climate change has emerged as a global challenge that affects weather patterns, agricultural productivity, water availability, and public health, requiring coordinated international efforts.",
    "Education plays a vital role in national development by empowering individuals with knowledge and skills, reducing socio-economic inequalities, and fostering innovation.",
    "Technological advancements in automation and artificial intelligence are reshaping industries, altering workforce requirements, and influencing economic productivity.",
    "Digital transformation in governance aims to improve transparency, efficiency, and citizen access to public services through technology-driven solutions.",
    "Healthcare accessibility remains a major concern in rural regions where limited infrastructure and workforce shortages impact service delivery.",
    "Language translation systems help bridge communication gaps between people from different linguistic backgrounds, enabling global collaboration.",
    "Machine learning models must be carefully evaluated to ensure fairness, reliability, and robustness across diverse real-world scenarios.",
    "The rapid growth of digital platforms has raised concerns related to data privacy, misinformation, and ethical use of technology.",
    "Renewable energy adoption is critical for reducing carbon emissions and ensuring long-term environmental sustainability.",
    "Education systems worldwide are adapting curricula to include digital literacy and critical thinking skills.",
    "Artificial Intelligence research continues to evolve as new models and training techniques are developed.",
    "Public health initiatives must consider social, economic, and cultural factors to achieve meaningful outcomes.",
    "Technological innovation plays a central role in shaping the future of global economies.",

    # Edge / stress-test cases (46–50)
    "In 2018, Google adopted Artificial Intelligence technologies to enhance search quality and user experience.",
    "The government announced a 25 percent increase in digital infrastructure spending in 2022.",
    "Artificial Intelligence, data analytics, and cloud computing are collectively driving Industry 4.0 initiatives.",
    "Healthcare policies must balance cost efficiency, service quality, and long-term sustainability.",
    "Effective language translation requires understanding context, grammar, and cultural nuances."
]


In [ ]:
#defining diffrent config for testing
decoding_configs = [
    {"temperature": 0.7, "top_k": 50,  "top_p": 0.9},
    {"temperature": 1.0, "top_k": 50,  "top_p": 0.9},
    {"temperature": 1.2, "top_k": 50,  "top_p": 0.9},
    {"temperature": 1.0, "top_k": 30,  "top_p": 0.8},
    {"temperature": 1.0, "top_k": 0,   "top_p": 1.0},  # greedy-ish
]


In [ ]:
## Define translation function
def translate(text, temperature, top_k, top_p):
    src_lang = "eng_Latn"
    tgt_lang = "hin_Deva"

    formatted_text = f"{src_lang} {tgt_lang} {text}"

    inputs = tokenizer(
        formatted_text,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
#running the function
results = []

for sentence_id, sentence in enumerate(test_sentences, start=1):
    for cfg_id, cfg in enumerate(decoding_configs, start=1):
        translation = translate(
            sentence,
            temperature=cfg["temperature"],
            top_k=cfg["top_k"],
            top_p=cfg["top_p"]
        )

        results.append({
            "Sentence_ID": sentence_id,
            "Source_Text": sentence,
            "Config_ID": cfg_id,
            "Temperature": cfg["temperature"],
            "Top_k": cfg["top_k"],
            "Top_p": cfg["top_p"],
            "Generated_Translation": translation
        })


In [ ]:
# saving results to csv
import pandas as pd
df = pd.DataFrame(results)
df.to_csv("indictrans2_decoding_evaluation_testing.csv", index=False)

df.head()


In [ ]:
#loading blue socre and rouge-score metrics
!pip install -q sacrebleu
!pip install -q rouge-score


In [ ]:
import evaluate

bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")


In [ ]:
## Load IIT Bombay test dataset
dataset = load_dataset("cfilt/iitb-english-hindi")
test_data = dataset["test"]

bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

preds, refs = [], []
MAX_SAMPLES = 2500

for i in range(MAX_SAMPLES):
    src = test_data[i]["translation"]["en"]
    tgt = test_data[i]["translation"]["hi"]

    pred = translate(src)
    preds.append(pred)
    refs.append([tgt])

bleu_score = bleu.compute(predictions=preds, references=refs)
rouge_score = rouge.compute(
    predictions=preds,
    references=[r[0] for r in refs]
)

print("BLEU:", bleu_score["score"])
print("ROUGE-1:", rouge_score["rouge1"])
print("ROUGE-2:", rouge_score["rouge2"])
print("ROUGE-L:", rouge_score["rougeL"])


In [ ]:
#saving result of all 2500 data into a csv file
import pandas as pd

rows = []

for i in range(MAX_SAMPLES):
    rows.append({
        "source_english": test_data[i]["translation"]["en"],
        "reference_hindi": test_data[i]["translation"]["hi"],
        "predicted_hindi": preds[i]
    })

df = pd.DataFrame(rows)

csv_path = "translation_results_2500_samples.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"CSV saved at: {csv_path}")
